In [ ]:
# The dataset contains around 1 million tweets and will be classified into positive and negative

In [15]:
!pip install pandas numpy nltk scikit-learn


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

print("All libraries imported")

All libraries imported


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [17]:
df = pd.read_csv(r"D:\Dataset for sentiment analysis\training.1600000.processed.noemoticon - Copy.csv", encoding="latin-1")
df.columns = ["label", "id", "date", "query", "username", "text"]  # rename columns
df = df[["label", "text"]] # show only label and text columns
print(df.head())

   label                                               text
0      0  is upset that he can't update his Facebook by ...
1      0  @Kenichan I dived many times for the ball. Man...
2      0    my whole body feels itchy and like its on fire 
3      0  @nationwideclass no, it's not behaving at all....
4      0                      @Kwesidei not the whole crew 


In [18]:
df_sample = df.sample(10000, random_state=42) # Sampling the dataset cause it's too large to process, would take so long

In [19]:
print(df_sample["label"].value_counts())

label
0    7658
4    2342
Name: count, dtype: int64


In [20]:
min_class_size = df_sample["label"].value_counts().min() # Balancing the sample

df_class_0 = df_sample[df_sample["label"] == 0].sample(min_class_size, random_state=42)
df_class_4 = df_sample[df_sample["label"] == 4].sample(min_class_size, random_state=42)

df_balanced = pd.concat([df_class_0, df_class_4]).sample(frac=1, random_state=42).reset_index(drop=True)

print(df_balanced["label"].value_counts())

label
0    2342
4    2342
Name: count, dtype: int64


In [21]:
# Initialization
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import re
from nltk.tokenize import word_tokenize

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()  # convert to lowercase
    text = re.sub(r'\W', ' ', text)  # remove special characters
    text = re.sub(r'\s+', ' ', text)  # remove extra spaces
    words = word_tokenize(text)
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]  # remove stopwords & lemmatize
    return " ".join(words)

df_balanced["cleaned_text"] = df_balanced["text"].apply(clean_text)
print(df_balanced.head())

   label                                               text  \
0      0  well i finally made it to sleep but woke up af...   
1      4              I am in the news  http://bit.ly/J27mM   
2      4  @TripCase honestly cannot stand the fact that ...   
3      4  @Jennybeeean oooo you're going on tuesday now,...   
4      4  @chromasia I saw your tweet minutes before goi...   

                                        cleaned_text  
0             well finally made sleep woke nightmare  
1                             news http bit ly j27mm  
2  tripcase honestly stand fact tripcase taken tw...  
3  jennybeeean oooo going tuesday fuuuunn much fu...  
4  chromasia saw tweet minute going lunch decided...  


In [22]:
print(df_balanced.columns)

X = df_balanced["cleaned_text"]
y = df_balanced["label"]

print("X and y defined successfully!")

Index(['label', 'text', 'cleaned_text'], dtype='object')
X and y defined successfully!


In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Data split successful!")

Data split successful!


In [24]:
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Text vectorization complete!")

Text vectorization complete!


In [25]:
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

print("Model training complete!")

Model training complete!


In [26]:
y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7171824973319103
              precision    recall  f1-score   support

           0       0.73      0.71      0.72       475
           4       0.71      0.73      0.72       462

    accuracy                           0.72       937
   macro avg       0.72      0.72      0.72       937
weighted avg       0.72      0.72      0.72       937

